
# Unsloth vs. Plain TRL — Speed Benchmark

**Day 2 — Tools & LLM Finetuning · Practical 3 of 3 · Companion to the "Tooling & Frameworks"
deck**

> **Running in Google Colab:** requires a **GPU runtime** (Runtime -> Change runtime type ->
> T4 GPU). This is exactly the workload Unsloth's official Colab notebooks are built around.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Finetune a small model with plain Hugging Face `transformers` + `peft` + TRL's `SFTTrainer`
2. Finetune the SAME model with Unsloth's `FastLanguageModel`, using the SAME TRL `SFTTrainer`
   underneath
3. Measure training time and peak VRAM directly, and compare against the deck's reported
   "2-4x faster, ~70% less VRAM" claim
4. See firsthand that Unsloth is an ACCELERATION layer under TRL, not a separate framework --
   the training loop code is nearly identical between the two paths

## Why This Matters for a Law Firm

If your firm is running finetuning jobs on a single GPU (a common constraint for legal-AI
tooling budgets, not a dedicated ML infrastructure team), the choice between plain TRL and
Unsloth has a direct, measurable cost and turnaround-time impact. This notebook lets you see
that impact on your own hardware rather than taking a vendor's benchmark on faith.

## Notebook Workflow

```mermaid
flowchart TD
    A["Same model + same\nlegal dataset"] --> B["Path 1: Plain TRL\n(transformers + peft + SFTTrainer)"]
    A --> C["Path 2: Unsloth\n(FastLanguageModel + SFTTrainer)"]
    B --> D["Compare: training time,\npeak VRAM"]
    C --> D



## Section 1 — Setup

We reuse the same small model and legal dataset pattern from the first notebook in this day,
so the only variable that changes between the two training paths is the loading/training
mechanism itself -- everything else (model choice, data, LoRA rank) stays identical.


In [ ]:

%pip install -q "unsloth[colab-new]" transformers accelerate peft bitsandbytes datasets trl

# Unsloth must be imported before transformers/peft/trl -- it patches those
# libraries on import, and importing it late causes eos_token/pad_token
# conflicts inside SFTConfig (ValueError: eos_token not found in vocabulary).
from unsloth import FastLanguageModel

import time
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")



## Section 2 — The Legal Dataset

Same small contract-boilerplate dataset used throughout this training, kept identical across
both paths so the comparison isolates the training MECHANISM, not the data.


In [ ]:

from datasets import Dataset

legal_sentences = [
    "This Agreement shall be governed by and construed in accordance with the laws of the State of Delaware.",
    "The Indemnitor shall indemnify and hold harmless the Indemnitee from any and all claims arising hereunder.",
    "Either party may terminate this Agreement upon thirty (30) days' prior written notice to the other party.",
    "The Receiving Party shall hold all Confidential Information in strict confidence.",
    "This Agreement constitutes the entire agreement between the parties and supersedes all prior negotiations.",
    "No waiver of any provision of this Agreement shall be effective unless in writing and signed by both parties.",
    "The parties agree that any dispute arising hereunder shall be resolved by binding arbitration.",
    "In the event of force majeure, neither party shall be liable for delay or failure to perform.",
    "This Agreement shall be binding upon and inure to the benefit of the parties and their successors.",
    "All notices under this Agreement shall be in writing and delivered to the addresses set forth above.",
    "The Effective Date of this Agreement is the date first written above.",
    "Each party represents and warrants that it has full power and authority to enter into this Agreement.",
]

raw_dataset = Dataset.from_dict({"text": legal_sentences})
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Training examples: {len(raw_dataset)}")



## Section 3 — Path 1: Plain TRL (transformers + peft + bitsandbytes)

This is the "before Unsloth" baseline: standard Hugging Face loading, `peft`'s `LoraConfig`,
and TRL's `SFTTrainer` -- exactly the deck's "TRL — Pseudocode" slide, made runnable.


In [ ]:

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

torch.cuda.reset_peak_memory_stats()

plain_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if plain_tokenizer.pad_token is None:
    plain_tokenizer.pad_token = plain_tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

plain_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

plain_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

plain_model = get_peft_model(plain_base_model, plain_lora_config)
plain_model.print_trainable_parameters()



## Section 4 — Train and Time Path 1


In [ ]:

plain_training_args = SFTConfig(
    output_dir="./plain-trl-legal",
    per_device_train_batch_size=4,
    num_train_epochs=15,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    report_to=[],
    max_length=128,   # renamed from max_seq_length (TRL >= 0.16.0)
)

plain_trainer = SFTTrainer(
    model=plain_model,
    train_dataset=raw_dataset,
    args=plain_training_args,
)

torch.cuda.synchronize()
plain_start = time.perf_counter()
plain_train_result = plain_trainer.train()
torch.cuda.synchronize()
plain_elapsed = time.perf_counter() - plain_start

plain_peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9

print(f"\nPlain TRL training time: {plain_elapsed:.1f} sec")
print(f"Plain TRL peak VRAM:     {plain_peak_vram_gb:.2f} GB")
print(f"Final training loss:     {plain_train_result.training_loss:.4f}")



## Section 5 — Path 2: Unsloth

Now the same model, same LoRA rank, same dataset, same TRL `SFTTrainer` -- but the model is
loaded through Unsloth's `FastLanguageModel` instead of plain `transformers`. Notice how little
the actual TRAINING code changes -- Unsloth accelerates what happens underneath, exactly as the
deck describes.


In [ ]:

# Free Path 1's model and reset memory tracking before loading Path 2
del plain_base_model, plain_model, plain_trainer
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# FastLanguageModel already imported at the top of the notebook (must come
# before transformers/trl -- see Section 1).
unsloth_model, unsloth_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=128,
    load_in_4bit=True,
)

unsloth_model = FastLanguageModel.get_peft_model(
    unsloth_model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
)

print("Unsloth model loaded and LoRA-configured.")



## Section 6 — Train and Time Path 2

The exact same `SFTTrainer` class from TRL, unchanged -- only the model object passed into it
is different (Unsloth-loaded vs. plain-`transformers`-loaded).


In [ ]:

unsloth_training_args = SFTConfig(
    output_dir="./unsloth-legal",
    per_device_train_batch_size=4,
    num_train_epochs=15,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    report_to=[],
    max_length=128,   # renamed from max_seq_length (TRL >= 0.16.0)
)

unsloth_trainer = SFTTrainer(
    model=unsloth_model,
    train_dataset=raw_dataset,
    args=unsloth_training_args,
)

torch.cuda.synchronize()
unsloth_start = time.perf_counter()
unsloth_train_result = unsloth_trainer.train()
torch.cuda.synchronize()
unsloth_elapsed = time.perf_counter() - unsloth_start

unsloth_peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9

print(f"\nUnsloth training time: {unsloth_elapsed:.1f} sec")
print(f"Unsloth peak VRAM:     {unsloth_peak_vram_gb:.2f} GB")
print(f"Final training loss:   {unsloth_train_result.training_loss:.4f}")



## Section 7 — The Comparison

The headline numbers, side by side.


In [ ]:

speedup = plain_elapsed / unsloth_elapsed
vram_reduction_pct = (1 - unsloth_peak_vram_gb / plain_peak_vram_gb) * 100

print(f"{'Metric':<25}{'Plain TRL':<15}{'Unsloth':<15}")
print("-" * 55)
print(f"{'Training time (sec)':<25}{plain_elapsed:<15.1f}{unsloth_elapsed:<15.1f}")
print(f"{'Peak VRAM (GB)':<25}{plain_peak_vram_gb:<15.2f}{unsloth_peak_vram_gb:<15.2f}")
print(f"{'Final loss':<25}{plain_train_result.training_loss:<15.4f}{unsloth_train_result.training_loss:<15.4f}")
print()
print(f"Speedup:        {speedup:.2f}x")
print(f"VRAM reduction: {vram_reduction_pct:.1f}%")



**Reading this result:** the deck's reported figures (roughly 2-4x faster, ~70% less VRAM) were
measured on larger models and longer training runs than this notebook's small demo -- at this
scale, expect a real but more modest speedup. The FINAL LOSS values should be very close between
the two paths -- this is the deck's "kernel rewrites, not algorithmic shortcuts" point made
visible: Unsloth computes the same math faster, it doesn't approximate a different result.



## Key Takeaways

1. **Unsloth genuinely is a drop-in acceleration layer** -- the training loop code (the
   `SFTTrainer` call itself) barely changed between the two paths; only the model-loading step
   did.
2. **Final training loss stayed comparable** between paths, confirming the deck's point that
   Unsloth's speedups come from kernel-level efficiency, not from computing a different,
   approximated result.
3. **The magnitude of the speedup scales with model size and training duration** -- a small
   model and a short demo run (like this notebook) will show a smaller gap than the deck's
   headline figures, which come from larger, longer, more realistic production training runs.
4. For a single-GPU finetuning setup -- the common case for a law firm without dedicated ML
   infrastructure -- this notebook is the concrete evidence for choosing Unsloth as the default,
   per the deck's "TRL vs. Unsloth — When to Use Which" guidance.

**This completes Day 2's hands-on practicals.** Combined with the Cursor and Claude Code decks'
live in-IDE demos, Day 2 covered the full path from choosing a finetuning approach (LoRA/QLoRA)
through the frameworks that actually run it (TRL, Unsloth) to compressing and serving the
result (quantization). **Next up:** Day 3 — RAG & Agents.
